In [ ]:
from __future__ import annotations

import json
import re
from collections import defaultdict
from pathlib import Path
from typing import Dict, List, Optional

import matplotlib.pyplot as plt
import numpy as np
import wandb

# ── W&B config ────────────────────────────────────────────────────────────────
import os
import wandb as _wandb_for_entity
# W&B entity: $WANDB_ENTITY if set, otherwise your default entity.
ENTITY  = os.environ.get("WANDB_ENTITY") or _wandb_for_entity.Api().default_entity
PROJECT = "appendix_autointerp"  # wandb.project in configs/appendix_autointerp/base.yaml

# ── Layout ────────────────────────────────────────────────────────────────────
GROUPS = ["lost", "control", "surviving"]
COLORS = {"lost": "C0", "control": "C1", "surviving": "C2"}
LABELS = {"lost": "Lost", "control": "Control", "surviving": "Surviving"}

MODEL_ORDER = [
    "gemma-3-270m",
    "gemma-3-1b-pt",
    "gemma-3-4b-pt",
    "gemma-3-12b-pt",
    "gemma-3-27b-pt",
    "gpt2-small",
    "meta-llama/Llama-3.1-8B",
    "mistralai/Mistral-7B-v0.1",
]

# Fixed column order for Gemma-Scope SAEs; non-Gemma models fill left-to-right
GEMMA_SAE_COLS = ["16k big", "16k medium", "16k small", "262k big", "262k medium", "262k small"]
N_COLS = 6

OUTPUT_DIR = Path("wandb_autointerp_grids")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

LABEL_FONTSIZE = 11
TITLE_FONTSIZE = 10
TICK_FONTSIZE  = 9

In [ ]:
# ── Helpers ───────────────────────────────────────────────────────────────────

def cfg_get(cfg, *keys, default=None):
    cur = cfg
    for k in keys:
        if isinstance(cur, dict) and k in cur:
            cur = cur[k]
        else:
            return default
    return cur


def pretty_model_name(model_name: str) -> str:
    return {
        "gemma-3-270m":              "Gemma 3 270M",
        "gemma-3-1b-pt":             "Gemma 3 1B",
        "gemma-3-4b-pt":             "Gemma 3 4B",
        "gemma-3-12b-pt":            "Gemma 3 12B",
        "gemma-3-27b-pt":            "Gemma 3 27B",
        "gpt2-small":                "GPT-2 Small",
        "meta-llama/Llama-3.1-8B":   "Llama 3.1 8B",
        "mistralai/Mistral-7B-v0.1": "Mistral 7B",
    }.get(model_name, model_name)


def get_model_name(run) -> Optional[str]:
    cfg = run.config or {}
    blob = " ".join([
        str(cfg_get(cfg, "sae", "model", "name") or ""),
        str(cfg_get(cfg, "sae", "release")        or ""),
        str(getattr(run, "group", "")             or ""),
    ]).lower()

    for model in MODEL_ORDER:
        if model.lower() in blob:
            return model

    if "270m"  in blob and "gemma" in blob: return "gemma-3-270m"
    if "1b-pt" in blob and "gemma" in blob: return "gemma-3-1b-pt"
    if "4b-pt" in blob and "gemma" in blob: return "gemma-3-4b-pt"
    if "12b"   in blob and "gemma" in blob: return "gemma-3-12b-pt"
    if "27b"   in blob and "gemma" in blob: return "gemma-3-27b-pt"
    if "gpt2"  in blob:                     return "gpt2-small"
    if "llama" in blob and "8b" in blob:    return "meta-llama/Llama-3.1-8B"
    if "mistral" in blob:                   return "mistralai/Mistral-7B-v0.1"
    return None


def get_sae_label(run) -> Optional[str]:
    cfg      = run.config or {}
    id_tmpl  = str(cfg_get(cfg, "sae", "id_template") or "")
    release  = str(cfg_get(cfg, "sae", "release")     or "").lower()
    run_name = str(getattr(run, "name", "")            or "")

    # Search id_template first, then run name (run names often ARE the id_template), then release
    for src in (id_tmpl, run_name, release):
        # Gemma-Scope: width_16k_l0_medium etc.
        m = re.search(r"width_(\d+k)_l0_(big|small|medium)", src)
        if m:
            return f"{m.group(1)} {m.group(2)}"

    for src in (id_tmpl, run_name, release):
        # GPT-2: gpt2-small-resid-post-v5-32k / -128k
        m = re.search(r"gpt2-small-resid-post-v\d+-(\d+k)", src)
        if m:
            return m.group(1)

    combined = " ".join([id_tmpl, run_name, release])
    # Llama scope
    if "lxr_32x" in combined or "r_32x" in combined: return "128k"  # 2**17 = 131,072 latents; called 128k by the release, like GPT-2 v5-128k
    if "lxr_8x"  in combined or "r_8x"  in combined: return "32k"

    # Mistral
    if "mistral-7b-res-wg" in combined: return "65k"

    return None


def sae_sort_key(label: str) -> tuple:
    m = re.match(r"(\d+)k\s*(big|medium|small)?", label)
    if m:
        size  = int(m.group(1))
        order = {"big": 0, "medium": 1, "small": 2}.get(m.group(2) or "", 3)
        return (size, order)
    return (999, 999)


def model_sort_key(name: str) -> int:
    return MODEL_ORDER.index(name) if name in MODEL_ORDER else 999


def extract_scores(run) -> Dict[str, Dict[int, Dict[str, float]]]:
    """Returns {group: {layer: {'score_mean': float, 'score_std': float}}}"""
    try:
        summary = dict(run.summary)
    except Exception:
        return {g: {} for g in GROUPS}

    result = {g: {} for g in GROUPS}
    for key, val in summary.items():
        # Handle keys with or without the "autointerp/" prefix that wandb.log() adds
        m = re.match(r"(?:autointerp/)?(lost|control|surviving)/layer_(\d+)/(score_mean|score_std)$", str(key))
        if not m:
            continue
        group, layer, metric = m.group(1), int(m.group(2)), m.group(3)
        if layer not in result[group]:
            result[group][layer] = {}
        try:
            result[group][layer][metric] = float(val)
        except (TypeError, ValueError):
            pass
    return result

In [ ]:
# ── Fetch runs from W&B ───────────────────────────────────────────────────────

api  = wandb.Api(timeout=60)
runs = list(api.runs(f"{ENTITY}/{PROJECT}"))
print(f"Found {len(runs)} runs in {ENTITY}/{PROJECT}")

# Keep only the latest run per (model, sae_label)
grouped: Dict[tuple, list] = defaultdict(list)
for run in runs:
    model = get_model_name(run)
    label = get_sae_label(run)
    if model and label:
        grouped[(model, label)].append(run)
    else:
        print(f"  skipped run '{getattr(run, 'name', '?')}': model={model}, label={label}")

run_map = {
    k: sorted(v, key=lambda r: str(getattr(r, "created_at", "")))[-1]
    for k, v in grouped.items()
}

# Extract scores
data: Dict[str, Dict[str, Dict]] = defaultdict(dict)
for (model, label), run in run_map.items():
    scores = extract_scores(run)
    n_scored = sum(len(v) for v in scores.values())
    print(f"  {pretty_model_name(model):20s}  {label:15s}  layers with data: {n_scored // len(GROUPS)}")
    if n_scored > 0:
        data[model][label] = scores

models_with_data = sorted(data.keys(), key=model_sort_key)
print(f"\nModels ready to plot: {[pretty_model_name(m) for m in models_with_data]}")

In [ ]:
# ── Plot helpers ──────────────────────────────────────────────────────────────

def draw_blank(ax) -> None:
    ax.set_visible(False)


def plot_interp_bars(
    ax,
    scores: Dict[str, Dict[int, Dict[str, float]]],
    sae_label: str,
    show_ylabel: bool = False,
    show_xlabel: bool = False,
) -> None:
    layers = sorted({l for g in GROUPS for l in scores[g].keys()})
    if not layers:
        draw_blank(ax)
        return

    x     = np.arange(len(layers))
    width = 0.22
    offsets = {"lost": -width, "control": 0.0, "surviving": width}

    for group in GROUPS:
        means = [scores[group].get(l, {}).get("score_mean", float("nan")) for l in layers]
        stds  = [scores[group].get(l, {}).get("score_std",  0.0)          for l in layers]
        ax.bar(
            x + offsets[group], means, width,
            yerr=stds, label=LABELS[group],
            color=COLORS[group], capsize=3, alpha=0.85,
            error_kw={"linewidth": 0.8},
        )

    ax.set_xticks(x)
    ax.set_xticklabels([str(l) for l in layers], fontsize=TICK_FONTSIZE)
    ax.set_ylim(0.5, 1.0)
    ax.set_yticks([0.5, 0.6, 0.7, 0.8, 0.9, 1.0])
    ax.tick_params(axis="y", labelsize=TICK_FONTSIZE)
    ax.yaxis.set_tick_params(labelleft=show_ylabel)
    ax.grid(axis="y", linewidth=0.4, alpha=0.4)
    ax.set_title(sae_label, fontsize=TITLE_FONTSIZE, pad=3)
    if show_ylabel:
        ax.set_ylabel("Autointerp score", fontsize=TICK_FONTSIZE)
    if show_xlabel:
        ax.set_xlabel("Layer", fontsize=TICK_FONTSIZE)


def get_col_layout(model: str, available_labels: List[str]) -> List[Optional[str]]:
    """Return length-N_COLS list: SAE label or None for empty panels."""
    if "gemma" in model.lower():
        return [lbl if lbl in available_labels else None for lbl in GEMMA_SAE_COLS]
    else:
        sorted_labels = sorted(available_labels, key=sae_sort_key)
        return sorted_labels + [None] * (N_COLS - len(sorted_labels))

In [ ]:
# ── Main grid ─────────────────────────────────────────────────────────────────

n_rows = len(models_with_data)
if n_rows == 0:
    print("No data to plot yet — run is still in progress or summary not written.")
else:
    fig, axes = plt.subplots(
        n_rows, N_COLS,
        figsize=(N_COLS * 2.8, n_rows * 2.5),
        squeeze=False,
    )

    for row_idx, model in enumerate(models_with_data):
        available   = list(data[model].keys())
        col_layout  = get_col_layout(model, available)
        is_last_row = row_idx == n_rows - 1

        # Row label on the left
        axes[row_idx, 0].annotate(
            pretty_model_name(model),
            xy=(0, 0.5), xycoords="axes fraction",
            xytext=(-0.55, 0.5), textcoords="axes fraction",
            ha="center", va="center",
            fontsize=LABEL_FONTSIZE, fontweight="bold",
            rotation=90, annotation_clip=False,
        )

        for col_idx, sae_label in enumerate(col_layout):
            ax = axes[row_idx, col_idx]
            if sae_label is None or sae_label not in data[model]:
                draw_blank(ax)
            else:
                plot_interp_bars(
                    ax,
                    data[model][sae_label],
                    sae_label,
                    show_ylabel=(col_idx == 0),
                    show_xlabel=is_last_row,
                )

    # Shared legend at the top
    handles, labels = [], []
    for ax in axes.flat:
        h, l = ax.get_legend_handles_labels()
        if h:
            handles, labels = h, l
            break

    fig.legend(
        handles, labels,
        loc="lower center", ncol=3,
        fontsize=LABEL_FONTSIZE, framealpha=0.9,
        bbox_to_anchor=(0.5, 1.0),
    )

    fig.tight_layout(rect=[0.04, 0.02, 1, 0.96])

    for fmt in ("pdf", "png"):
        p = OUTPUT_DIR / f"autointerp_grid.{fmt}"
        fig.savefig(p, bbox_inches="tight", dpi=200)
        print(f"Saved: {p}")

    plt.show()